<a href="https://colab.research.google.com/github/laisalegri/curso-fiap-agentes-autonomos-ia/blob/main/FIAP_hands_on.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain -q
!pip install langchain-core -q
!pip install langchain-community -q
!pip install langchain-ollama -q
!pip install langgraph -q

In [ ]:
import langchain
import langchain_core
import langchain_community
import langgraph
import importlib.metadata

print("LangChain version:", langchain.__version__)
print("LangChain Core version:", langchain_core.__version__)
print("LangChain Community version:", langchain_community.__version__)

try:
    langgraph_version = importlib.metadata.version('langgraph')
    print("LangGraph version:", langgraph_version)
except importlib.metadata.PackageNotFoundError:
    print("LangGraph version: Installed but package metadata not found")

print("\nAll packages imported successfully!")

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict, Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage

# Define o schema do estado
class State(TypedDict):
    messages: Annotated[list[HumanMessage], add_messages]

# Define a função do nó de entrada
def entrada_usuario(state: State) -> State:
  pergunta = input("Pergunta: ")
  print(f"Entrada recebida: {pergunta}")
  if isinstance(pergunta, str) and pergunta.strip():
    # HumanMessage object
    return {"messages": [HumanMessage(content=pergunta)]}
  else:
    raise ValueError("A pergunta deve ser uma string nao vazia.")

# Define to no de processamento da resposta
def processamento_resposta(state: State) -> State:
  pergunta = state["messages"][-1].content
  print(f"Estado na funcao de processamento: {state}")

  if isinstance(pergunta, str) and pergunta.strip():
    # Resposta simulada
    resposta = f"A resposta para a sua pergunta ' {pergunta}' e: [resposta simulada]"
    print(f"Resposta gerada: {resposta}")
    return {"messages": [resposta]}
  else:
    raise ValueError("A pergunta deve ser uma string não vazia.")

# Define the output node function
def saida_resposta(state: State) -> State:
  print(f"Estado na função de saida: {state}")
  print(state["messages"] [-1])
  return state

# Create the state graph
grafo = StateGraph(State)

# Add nodes to the graph
grafo.add_node("entrada", entrada_usuario)
grafo.add_node("processamento", processamento_resposta)
grafo.add_node("saida", saida_resposta)

# Define the connections between nodes
grafo.add_edge(START, "entrada")
grafo.add_edge("entrada", "processamento")
grafo.add_edge("processamento", "saida")
grafo.add_edge("saida", END)

# Compile the graph
compiled = grafo.compile()

# Implement the interaction loop
while True:
  compiled.invoke({"messages": [HumanMessage(content="Como posso te ajudar hoje?")]})
  continuar = input("Você deseja fazer outra pergunta? (sim/não): ")
  if continuar.lower() != 'sim':
    break

In [ ]:
from IPython.display import Image

Image(compiled.get_graph().draw_mermaid_png())

In [ ]:
import langgraph
import numpy as np
import pandas as pd

In [ ]:
# 1. Instala o zstd e depois instala o Ollama
!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Inicia o servidor do Ollama em background
!nohup ollama serve > /dev/null 2>&1 &

# 3. Faz o download do modelo
!ollama pull gpt-oss:20b

In [ ]:
from langchain_community.llms import Ollama
#from langchain_ollama import OllamaLLM

# Testando o 11m
def processamento_resposta(pergunta):
  # Initialize the ollama LLM
  llm = Ollama(model="gpt-oss:20b",num_predict=1024)

  # Send a prompt
  resposta = llm.invoke(pergunta)
  return resposta

processamento_resposta("Diga: Hello World!")

'Hello World!'

In [ ]:
!pip install -U langchain-ollama

In [ ]:
!pgrep -af ollama
!curl -s http://127.0.0.1:11434/api/tags

9629 ollama serve
{"models":[{"name":"gpt-oss:20b","model":"gpt-oss:20b","modified_at":"2026-11-05T14:15:43.612558071Z","size":13793441566,"digest":"f38aa0c53da5f8c49d08c43a99df24ff53167fe68e24664a7777288e7656fdfe","details":{"parent_model":"dhiltgen/gpt-oss:20b-llamacpp","format":"gguf","family":"gpt-oss","families":["gpt-oss"],"parameter_size":"20.9B","quantization_level":"MXFP4","context_length":131072,"embedding_length":2880,"runner":"llamacpp"},"capabilities":["completion","tools","thinking"]}]}

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict, Annotated
from langchain_community.llms import Ollama
from langchain_core.messages import HumanMessage, AIMessage

# Inicializar o modelo ollama
llm = Ollama(model="gpt-oss:20b", num_predict=1024)

# Definir o schema de estado, mantendo historico de mensagens
class State(TypedDict):
  # usamos Annotated + add_messages para indicar que devemos *adicionar* a lista de mensagens
  messages: Annotated[list[HumanMessage | AIMessage], add_messages]

# Funcão de entrada: lê o input do usuario, valida nao-vazio e adiciona ao histórico
def entrada_usuario(state: State) -> State:
    pergunta_texto = input("Pergunta: ").strip()
    if not pergunta_texto:
      # Caso o usuário não digite nada ou apenas espaços
      raise ValueError("A pergunta deve ser uma string nao-vazia.")
    pergunta_msg = HumanMessage(content=pergunta_texto)
    return {"messages": [pergunta_msg]}

# Funcão de processamento: responde à ultima mensagem humana e adiciona a resposta
def processamento_resposta(state: State) -> State:
    # assumimos que a última mensagem é humana
    last_msg = state["messages"][-1]
    if not isinstance(last_msg, HumanMessage):
        # se por acaso não for humana, tratar como erro
        raise ValueError("Ultima mensagem esperada como HumanMessage.")
    pergunta = last_msg.content
    resposta_texto = llm.invoke(pergunta)
    resposta_msg = AIMessage(content=f"A resposta para a sua pergunta '{pergunta}' e: {resposta_texto}")
    return {"messages": [resposta_msg]}

# Funcão de saída: imprime a última resposta e não altera o estado (ou poderia adicionar um sistema de log)
def saida_resposta(state: State) -> State:
    last_msg = state["messages"][-1]
    if not isinstance(last_msg, AIMessage):
        raise ValueError("Última mensagem esperada como AIMessage.")
    print(last_msg.content)
    # Como nao queremos descartar o historico, retornamos estado vazio de atualizacão (nenhuma modificação)
    return {}

# Criar o grafo de estado
grafo = StateGraph(State)

# Adicionar nós
grafo.add_node("entrada", entrada_usuario)
grafo.add_node("processamento", processamento_resposta)
grafo.add_node("saida", saida_resposta)

# Definir arestas (fluxo linear para este caso simples)
grafo.add_edge(START, "entrada")
grafo.add_edge("entrada", "processamento")
grafo.add_edge("processamento", "saida")
grafo.add_edge("saida", END)

#Compilar o grafo
compiled = grafo.compile()

# Implementar ciclo de interação
# Inicializamos com histórico vazio
initial_state: State = {"messages": []}

while True:
    try:
        # Passamos o estado acumulado
        result_state = compiled.invoke(initial_state)
        # Atualizamos o estado com o retorno (as mensagens sao acumuladas automaticamente via Annotated + add_messages)
        # Em LangGraph, a invocacao retorna o novo estado completo, então:
        initial_state = result_state
    except Exception as e:
        print("Erro durante a execução:", e)
        # Opcional: decidir se continua ou sai
        break

    continuar = input("Você deseja fazer outra pergunta? (sim/não): ").strip().lower()
    if continuar != 'sim':
        break

print("Encerrando. Historico de mensagens:")
for msg in initial_state["messages"]:
    role = "Usuário" if isinstance(msg, HumanMessage) else "Assistente"
    print(f"[{role}] {msg.content}")


Pergunta: Quanto é 1 + 1?
A resposta para a sua pergunta 'Quanto é 1 + 1?' e: 1 + 1 = 2.
Você deseja fazer outra pergunta? (sim/não): não
Encerrando. Historico de mensagens:
[Usuario] Quanto é 1 + 1?
[Assistente] A resposta para a sua pergunta 'Quanto é 1 + 1?' e: 1 + 1 = 2.
